# Dataset Exploration — Motor Insurance Portfolio

DOI: 10.17632/sw4jmdb2sm.1  
354,140 records × 47 variables  
**Target:** `total_premium` (Gamma GLM)

**Goal:** understand the data well enough to configure `config/project_config.yaml`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_PATH = '../data/Dataset of motor insurance portfolio.csv'
SEP = ';'  # CSV uses semicolon separator

## 1. Load & shape

In [2]:
df = pd.read_csv(DATA_PATH, sep=SEP, low_memory=False)
print(f"Shape: {df.shape}")
df.head(3)

Shape: (354140, 47)


,insured_id,year,policy_type,policy_status,business_type,payment_frequency,bonus_score,driver_age,vehicle_age,age_driving_licence,fuel_type,vehicle_value,seats,power_to_weight_ratio,vehicle_brand,municipality_type,circulation_area,total_premium,liability_premium,property_damage_premium,theft_premium,fire_premium,glass_premium,legal_protection_premium,occupants_premium,total_claims,liability_claims,liability_property_claims,liability_injury_claims,property_claims,theft_claims,fire_claims,glass_claims,legal_protection_claims,occupants_claims,total_incurred,liability_incurred,liability_property_incurred,liability_injury_incurred,property_incurred,theft_incurred,fire_incurred,glass_incurred,legal_protection_incurred,occupants_incurred,total_exposure,liability_exposure
0,1,2022,COMP_E,C,NB,A,G,48,22.0000,26.0000,G,149511.4425,4,3.9200,PORSCHE,I,U,279.5576,30.6023,121.9127,112.3615,3.8324,8.3058,2.0436,0.4992,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0986,0.0986
1,2,2022,COMP_E,A,P,A,G,43,24.0000,19.0000,D,65065.1232,8,10.1500,MERCEDES,I,U,546.6234,114.3334,303.1060,80.7848,7.8269,30.0918,7.8127,2.6679,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000
2,2,2023,COMP_E,A,P,A,G,44,25.0000,19.0000,D,65065.1232,8,10.1500,MERCEDES,I,U,548.7526,110.6669,304.9560,83.5214,6.1515,32.8616,7.6556,2.9396,1,1,1,0,0,0,0,0,0,0,233.0130,233.0130,233.0130,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000


## 2. Column overview — dtypes, nulls, unique counts

In [3]:
overview = pd.DataFrame({
    'dtype': df.dtypes,
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().mean() * 100).round(2),
    'n_unique': df.nunique(),
    'sample': [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
overview

,dtype,null_count,null_pct,n_unique,sample
insured_id,int64,0,0.0000,185678,1
year,int64,0,0.0000,3,2022
policy_type,str,0,0.0000,5,COMP_E
policy_status,str,0,0.0000,2,C
business_type,str,0,0.0000,2,NB
payment_frequency,str,0,0.0000,3,A
bonus_score,str,0,0.0000,3,G
driver_age,int64,0,0.0000,73,48
vehicle_age,float64,2,0.0000,68,22.0000
age_driving_licence,float64,2,0.0000,71,26.0000


## 3. Target variable — total_premium

In [4]:
target_col   = 'total_premium'
exposure_col = 'total_exposure'

print(f"Zeros:     {(df[target_col] == 0).sum():,}  ({(df[target_col] == 0).mean()*100:.3f}%)")
print(f"Nulls:     {df[target_col].isnull().sum():,}")
print(f"Negatives: {(df[target_col] < 0).sum():,}")
print()
print("Distribution:")
print(df[target_col].describe())

# 49 zeros (0.014%) — negligible, likely cancelled policies; filter for modelling
df_model = df[df[target_col] > 0].copy()
print(f"\nRows after removing zeros: {len(df_model):,}")

Zeros:     49  (0.014%)
Nulls:     0
Negatives: 0

Distribution:
count   354140.0000
mean       296.7179
std        227.8548
min          0.0000
25%        146.1447
50%        257.9031
75%        384.9806
max       5107.6399
Name: total_premium, dtype: float64

Rows after removing zeros: 354,091


## 4. Exposure

In [5]:
print(f"Total exposure (policy-years): {df_model[exposure_col].sum():,.0f}")
print(f"Mean premium per policy-year:  {(df_model[target_col] / df_model[exposure_col]).mean():.2f}")
print()
print("Exposure distribution:")
df_model[exposure_col].describe()

Total exposure (policy-years): 250,326
Mean premium per policy-year:  420.01

Exposure distribution:


count   354091.0000
mean         0.7070
std          0.3262
min          0.0027
25%          0.4438
50%          0.8384
75%          1.0000
max          1.0000
Name: total_exposure, dtype: float64

## 5. Numeric features — distributions

In [6]:
numeric_candidates = [
    'driver_age', 'vehicle_age', 'age_driving_licence',
    'vehicle_value', 'seats', 'power_to_weight_ratio',
]

df_model[numeric_candidates].describe().T

,count,mean,std,min,25%,50%,75%,max
driver_age,354091.0000,49.0054,12.0687,18.0000,39.0000,48.0000,58.0000,90.0000
vehicle_age,354089.0000,25.9210,11.6173,0.0000,17.0000,25.0000,35.0000,68.0000
age_driving_licence,354089.0000,22.9905,6.3594,0.0000,19.0000,20.0000,25.0000,80.0000
vehicle_value,353578.0000,26977.5480,12326.2214,1630.1250,18607.5750,24729.6000,32252.3250,374034.2098
seats,354091.0000,5.0840,0.7570,2.0000,5.0000,5.0000,5.0000,9.0000
power_to_weight_ratio,354091.0000,12.3339,2.4823,0.0000,10.7400,12.0500,13.7300,64.8100


## 6. Categorical features — value counts

In [7]:
categorical_candidates = [
    'policy_type', 'policy_status', 'business_type',
    'payment_frequency', 'bonus_score', 'fuel_type',
    'vehicle_brand', 'municipality_type', 'circulation_area',
]

for col in categorical_candidates:
    if col not in df_model.columns:
        continue
    vc = df_model[col].value_counts(dropna=False)
    print(f"\n--- {col} ({df_model[col].nunique()} unique) ---")
    print(vc.head(15).to_string())


--- policy_type (5 unique) ---
policy_type
CC        204609
COMP_E     91186
TPG        36358
COMP_N     16181
TP          5757

--- policy_status (2 unique) ---
policy_status
A    315812
C     38279

--- business_type (2 unique) ---
business_type
NB    245425
P     108666

--- payment_frequency (3 unique) ---
payment_frequency
A    287859
S     50837
Q     15395

--- bonus_score (3 unique) ---
bonus_score
G    341511
N      7850
B      4730

--- fuel_type (2 unique) ---
fuel_type
D      226367
G      126437
NaN      1287

--- vehicle_brand (68 unique) ---
vehicle_brand
RENAULT       34348
CITROEN       32486
PEUGEOT       29569
VOLKSWAGEN    28785
FORD          28294
OPEL          27939
SEAT          27930
AUDI          16279
MERCEDES      15544
TOYOTA        14891
BMW           13825
NISSAN        13422
HYUNDAI        9288
FIAT           6978
KIA            6144

--- municipality_type (3 unique) ---
municipality_type
I     228580
C     101374
IS     24137

--- circulation_area (2 un

## 7. Mean premium by key categoricals

In [8]:
def premium_by(col):
    g = df_model.groupby(col, dropna=False).agg(
        exposure=(exposure_col, 'sum'),
        mean_premium=(target_col, 'mean'),
        median_premium=(target_col, 'median'),
        n_policies=(target_col, 'count'),
    )
    g['exposure_pct'] = (g['exposure'] / g['exposure'].sum() * 100).round(1)
    return g.sort_values('mean_premium', ascending=False)

for col in ['policy_type', 'bonus_score', 'fuel_type', 'municipality_type', 'circulation_area']:
    if col not in df_model.columns:
        continue
    print(f"\n=== Premium by {col} ===")
    print(premium_by(col).to_string())


=== Premium by policy_type ===
               exposure  mean_premium  median_premium  n_policies  exposure_pct
policy_type                                                                    
COMP_N       12342.3507      639.3023        601.0793       16181        4.9000
COMP_E       65040.3370      391.6434        380.3312       91186       26.0000
CC          144235.1863      243.3997        242.3601      204609       57.6000
TPG          24905.3507      225.5737        215.3741       36358        9.9000
TP            3802.6740      177.0973        160.8862        5757        1.5000

=== Premium by bonus_score ===
               exposure  mean_premium  median_premium  n_policies  exposure_pct
bonus_score                                                                    
B             3078.2712      589.0242        448.7030        4730        1.2000
N             4878.3452      416.3937        293.3566        7850        1.9000
G           242369.2822      289.9611        256.4953   

## 8. Coverage-level premiums — which sub-coverages drive the total?

In [9]:
premium_cols = [
    'liability_premium', 'property_damage_premium', 'theft_premium',
    'fire_premium', 'glass_premium', 'legal_protection_premium', 'occupants_premium',
]

rows = []
total = df_model['total_premium'].sum()
for col in premium_cols:
    if col not in df_model.columns:
        continue
    s = df_model[col].sum()
    rows.append({
        'coverage': col.replace('_premium', ''),
        'total': round(s, 0),
        'share_pct': round(s / total * 100, 2),
        'mean': round(df_model[col].mean(), 2),
        'pct_zero': round((df_model[col] == 0).mean() * 100, 1),
    })

pd.DataFrame(rows).set_index('coverage').sort_values('share_pct', ascending=False)

,total,share_pct,mean,pct_zero
coverage,,,,
liability,63997611.0000,60.9000,180.7400,0.0000
property_damage,26034961.0000,24.7800,73.5300,69.1000
glass,6904689.0000,6.5700,19.5000,1.6000
theft,3983622.0000,3.7900,11.2500,11.9000
legal_protection,2927937.0000,2.7900,8.2700,0.0000
occupants,695915.0000,0.6600,1.9700,0.0000
fire,534934.0000,0.5100,1.5100,11.9000


## 9. Driving licence experience — derive from calendar year

In [10]:
# age_driving_licence is the calendar year of issue, not the duration
if 'age_driving_licence' in df_model.columns and 'year' in df_model.columns:
    df_model['licence_experience'] = df_model['year'] - df_model['age_driving_licence']
    print("Licence experience (years):")
    print(df_model['licence_experience'].describe())
    print(f"\nNegative values (data quality): {(df_model['licence_experience'] < 0).sum()}")

Licence experience (years):
count   354089.0000
mean      2000.2945
std          6.4055
min       1944.0000
25%       1998.0000
50%       2003.0000
75%       2004.0000
max       2024.0000
Name: licence_experience, dtype: float64

Negative values (data quality): 0


## 10. vehicle_brand — cardinality check

In [11]:
if 'vehicle_brand' in df_model.columns:
    brand_stats = premium_by('vehicle_brand')
    print(f"Unique brands: {len(brand_stats)}")
    print("\nTop 20 by mean premium:")
    print(brand_stats.head(20).to_string())
    print("\nSmallest brands by exposure:")
    print(brand_stats.sort_values('exposure').head(10).to_string())

Unique brands: 68

Top 20 by mean premium:
                exposure  mean_premium  median_premium  n_policies  exposure_pct
vehicle_brand                                                                   
ASTON MARTIN      3.8110     1491.1827       1806.3090           4        0.0000
FERRARI           4.0740     1449.8373       1119.6990           6        0.0000
MASERATI          4.9425     1016.7923       1052.0079           6        0.0000
INFINITI         26.8055      753.2165        664.7462          36        0.0000
BENTLEY           0.5562      748.0617        748.0617           1        0.0000
PORSCHE         235.4000      650.8348        517.6784         327        0.1000
LEXUS           596.8521      504.4195        419.1236         833        0.2000
JAGUAR          443.7616      455.0881        370.4583         612        0.2000
AIXAM             7.9123      451.0893        495.1370           9        0.0000
IVECO            84.5781      436.3886        390.6619         114

## 11. Candidate YAML config — review before pasting into project_config.yaml

In [12]:
yaml_draft = """
# ── CANDIDATE CONFIG — review & adjust before using ──────────────────────────
data:
  path: "data/Dataset of motor insurance portfolio.csv"
  sep: ";"                          # semicolon-delimited CSV
  target_col: "total_premium"       # Gamma GLM; 49 zero-premium rows filtered out
  exposure_col: "total_exposure"
  objective: "gamma"

features:
  numeric:
    - name: "driver_age"
      description: "Age of the main driver in years"
    - name: "vehicle_age"
      description: "Age of the insured vehicle in years"
    - name: "licence_experience"   # derived: year - age_driving_licence
      description: "Years since the driver obtained their licence"
    - name: "vehicle_value"
      description: "Declared insured value of the vehicle"
    - name: "power_to_weight_ratio"
      description: "kg per horsepower — proxy for vehicle performance"
    - name: "seats"
      description: "Number of seats in the vehicle"

  categorical:
    - name: "policy_type"
      description: "Coverage structure: TP / TPG / CC / COMP_E / COMP_N"
      n_clusters: 3   # TP-like / intermediate / comprehensive
    - name: "bonus_score"
      description: "Claims history indicator: G (good) / N (neutral) / B (bad)"
      n_clusters: 3
    - name: "fuel_type"
      description: "D (diesel) or G (gasoline)"
      n_clusters: 2
    - name: "municipality_type"
      description: "I (inland) / C (coastal) / IS (islands)"
      n_clusters: 3
    - name: "circulation_area"
      description: "U (urban) or R (rural)"
      n_clusters: 2
    - name: "vehicle_brand"
      description: "Manufacturer brand — high cardinality, needs grouping"
      n_clusters: 6   # review cell 10 output before fixing this

llm:
  model: "claude-sonnet-4-6"
  temperature: 0.2
  max_hypotheses: 5

validation:
  objective: "gamma"
  n_rounds: 300
  early_stopping_rounds: 50
  min_deviance_improvement_pct: 0.1
  min_exposure_per_cluster: 500

output:
  report_path: "reports/"
  approval_log: "reports/actuary_decisions.csv"
"""

print(yaml_draft)


# ── CANDIDATE CONFIG — review & adjust before using ──────────────────────────
data:
  path: "data/Dataset of motor insurance portfolio.csv"
  sep: ";"                          # semicolon-delimited CSV
  target_col: "total_premium"       # Gamma GLM; 49 zero-premium rows filtered out
  exposure_col: "total_exposure"
  objective: "gamma"

features:
  numeric:
    - name: "driver_age"
      description: "Age of the main driver in years"
    - name: "vehicle_age"
      description: "Age of the insured vehicle in years"
    - name: "licence_experience"   # derived: year - age_driving_licence
      description: "Years since the driver obtained their licence"
    - name: "vehicle_value"
      description: "Declared insured value of the vehicle"
    - name: "power_to_weight_ratio"
      description: "kg per horsepower — proxy for vehicle performance"
    - name: "seats"
      description: "Number of seats in the vehicle"

  categorical:
    - name: "policy_type"
      description: "Coverag